[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sei-international/trase-notebooks/blob/main/public-data/explore-trase-data.ipynb)

# Explore Trase data

[Trase](https://trase.earth) links the places where commodities such as soy, palm oil or wood pulp are produced, and the deforestation there, with the companies that trade them and the countries that buy them. Here you can ask your own questions of its data, in Google Colab, with nothing to install.

To start, choose *Runtime → Run all*.

> **Demo.** This is a trial of a new way of publishing Trase data. Some descriptions are incomplete, and the data may change. For the official Trase data, go to [trase.earth](https://trase.earth).

## Connect

In [1]:
%pip install --quiet "duckdb>=1.5"

import duckdb

con = duckdb.connect()
con.execute(
    "ATTACH 'ducklake:https://public.deforestationfree.com/trase.ducklake' AS trase (READ_ONLY)"
)
con.execute("USE trase");

Note: you may need to restart the kernel to use updated packages.


## What's in it

Each dataset is one commodity in one country. Its `supply_chains` table has one row per trade flow and year; `spatial_metrics` has figures such as deforestation for each place and year, and `regions` the places and their boundaries.

In [2]:
con.sql("""
    SELECT schema_name AS dataset, table_name AS "table", comment AS description
    FROM duckdb_tables()
    WHERE database_name = 'trase'
    ORDER BY ALL
""").df()

,dataset,table,description
0,cote_divoire_cocoa_v1_2_0,regions,Côte d’Ivoire cocoa (cote_divoire_cocoa_v1_2_0...
1,cote_divoire_cocoa_v1_2_0,spatial_metrics,Côte d’Ivoire cocoa (cote_divoire_cocoa_v1_2_0...
2,cote_divoire_cocoa_v1_2_0,spatial_metrics_metadata,Côte d’Ivoire cocoa (cote_divoire_cocoa_v1_2_0...
3,cote_divoire_cocoa_v1_2_0,supply_chains,Côte d’Ivoire cocoa (cote_divoire_cocoa_v1_2_0...
4,cote_divoire_cocoa_v1_2_0,supply_chains_columns_metadata,Côte d’Ivoire cocoa (cote_divoire_cocoa_v1_2_0...
5,cote_divoire_cocoa_v1_2_0,supply_chains_context_metadata,Côte d’Ivoire cocoa (cote_divoire_cocoa_v1_2_0...
6,indonesia_wood_pulp_v3_2_0,regions,Indonesia wood pulp (indonesia_wood_pulp_v3_2_...
7,indonesia_wood_pulp_v3_2_0,spatial_metrics,Indonesia wood pulp (indonesia_wood_pulp_v3_2_...
8,indonesia_wood_pulp_v3_2_0,spatial_metrics_metadata,Indonesia wood pulp (indonesia_wood_pulp_v3_2_...
9,indonesia_wood_pulp_v3_2_0,supply_chains,Indonesia wood pulp (indonesia_wood_pulp_v3_2_...


## A first question

Which companies export the most deforestation from Indonesia, in wood pulp? For each pulp mill's company group, the deforestation embedded in its exports in 2024, in hectares:

In [3]:
con.sql("""
    SELECT
        pulp_mill_corporate_group AS company_group,
        round(sum(wood_pulp_deforestation_10_year_total_exposure)) AS deforestation_ha,
        round(sum(volume)) AS exports_tonnes
    FROM indonesia_wood_pulp_v3_2_0.supply_chains
    WHERE year = 2024 AND country_of_destination <> 'INDONESIA'
    GROUP BY company_group
    ORDER BY deforestation_ha DESC
""").df()

,company_group,deforestation_ha,exports_tonnes
0,ROYAL GOLDEN EAGLE,139928.0,2824185.0
1,SINAR MAS,66882.0,3191831.0
2,MARUBENI,0.0,425884.0


Change `2024` to another year, and run the cell again with ▶.

## Ask Gemini

You don't need to write the queries yourself. Click **✦ Gemini** at the bottom of the window, and ask in your own words, for example:

> Using `con`, which countries imported the most Indonesian wood pulp in 2024, and how much deforestation was embedded in their imports? Read the column descriptions first.

Gemini writes the code in a new cell: accept it, and run it. Check what it did against the column descriptions, which `duckdb_columns()` holds.

The first time, Colab shows a privacy notice: Google collects your prompts to Gemini, with the related code and output.

## Ideas to explore

- How has deforestation for pulpwood changed since 2001?
- Which provinces, or concessions, had the most? Draw them on a map.
- Where does Indonesian pulp go, and how has that changed?
- Do companies you know appear in the data? Look them up by name.
- Côte d'Ivoire cocoa is here too: ask the same questions of it.

## Go further

- [Indonesian wood pulp: worked examples](https://colab.research.google.com/github/sei-international/trase-notebooks/blob/main/public-data/indonesia-wood-pulp-examples.ipynb): where the pulp goes, who exports it, and the deforestation and emissions linked to it, with charts and a map.
- [Machine-friendly Trase Data](https://public.deforestationfree.com/index.html): every dataset, its files and its documentation.
- [trase.earth](https://trase.earth): Trase's insights, reports and tools.